# Forward Replay: Compare forward vs backward vs k-swap

Loads replay_results.json from the synthetic seed sweep. Compares three algorithms across random N∈{8..12} instances in R²/R³ over a gamma grid.

## Setup — load results

In [1]:
%matplotlib inline
import os, json
import numpy as np
import matplotlib.pyplot as plt

# works whether the notebook runs from notebooks/ or the repo root
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
RES = os.path.join(ROOT, "results", "forward_beats_backward")

payload = json.load(open(os.path.join(RES, "replay_results.json")))
records = payload["records"]
config = payload["config"]

print(f"Loaded {len(records)} records")
print(f"Config: {config}")

Loaded 100000 records
Config: {'n_seeds': 400, 'n_range': [8, 9, 10, 11, 12], 'dims': [2, 3], 'n_gamma': 25, 'metric': 'mean_abs', 'kswap_k': 2}


## Aggregate results by config and gamma

In [ ]:
# Group by config and gamma, compute statistics
configs = ["n_forward", "n_backward", "n_kswap"]
config_names = {"n_forward": "forward", "n_backward": "backward", "n_kswap": "k-swap"}

# Extract unique gammas
gammas = sorted(set(r["gamma"] for r in records))
print(f"Unique gammas: {len(gammas)}")
print(f"Gamma range: {min(gammas):.4f} to {max(gammas):.4f}")

# Aggregate
summary = {}
for cfg_key in configs:
    summary[cfg_key] = {}
    for g in gammas:
        sizes = [r[cfg_key] for r in records if r["gamma"] == g]
        summary[cfg_key][g] = {
            "mean_size": float(np.mean(sizes)),
            "std_size": float(np.std(sizes)),
            "min_size": int(np.min(sizes)),
            "max_size": int(np.max(sizes)),
            "n_instances": len(sizes),
        }

print("\nSummary keys:", list(summary.keys()))
print(f"Example (forward, first gamma): {summary['n_forward'][gammas[0]]}")

Unique gammas: 100000
Gamma range: 0.0172 to 2.0524


## Plot 1 — |S| vs γ (all configs, mean ± std)

In [1]:
x = np.array(gammas)
colors = {"n_forward": "blue", "n_backward": "red", "n_kswap": "green"}

fig, ax = plt.subplots(figsize=(9, 5))
for cfg_key in configs:
    m = np.array([summary[cfg_key][g]["mean_size"] for g in gammas])
    s = np.array([summary[cfg_key][g]["std_size"] for g in gammas])
    label = config_names[cfg_key]
    ax.plot(x, m, "-o", color=colors[cfg_key], lw=2, ms=4, label=label)
    ax.fill_between(x, m - s, m + s, color=colors[cfg_key], alpha=0.15)

ax.set_xlabel("tolerance γ", fontsize=12)
ax.set_ylabel("kept models |S|", fontsize=12)
ax.set_title(f"|S| vs γ  ({config['n_seeds']} seeds, N={config['n_range']}, dims={config['dims']}, mean ± std)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

NameError: name 'np' is not defined

## Plot 2 — Grouped bars: models kept per config × gamma

In [ ]:
ng, nc = len(gammas), len(configs)
w = 0.8 / nc
base = np.arange(ng)

fig, ax = plt.subplots(figsize=(12, 5))
for i, cfg_key in enumerate(configs):
    m = np.array([summary[cfg_key][g]["mean_size"] for g in gammas])
    s = np.array([summary[cfg_key][g]["std_size"] for g in gammas])
    label = config_names[cfg_key]
    ax.bar(base + i * w, m, w, yerr=s, capsize=2, color=colors[cfg_key],
           label=label, error_kw={"elinewidth": 0.7})

ax.set_xticks(base + 0.4 - w / 2)
ax.set_xticklabels([f"{g:.2f}" for g in gammas], rotation=45, fontsize=9)
ax.set_xlabel("tolerance γ", fontsize=12)
ax.set_ylabel("kept models |S|  (mean, error = std)", fontsize=12)
ax.set_title(f"Models kept per config  ({config['n_seeds']} random seeds)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## Plot 3 — Algorithm "wins" per gamma (smallest |S|)

In [ ]:
# Count wins (i.e., for each (seed, gamma, N, dim) which config has smallest |S|)
wins = {cfg_key: [] for cfg_key in configs}

# Group by (N, dim, seed, gamma)
for g in gammas:
    rows_at_g = [r for r in records if r["gamma"] == g]
    win_counts = {cfg_key: 0 for cfg_key in configs}
    
    for row in rows_at_g:
        sizes = [row[cfg_key] for cfg_key in configs]
        best_cfg = configs[np.argmin(sizes)]
        win_counts[best_cfg] += 1
    
    for cfg_key in configs:
        wins[cfg_key].append(win_counts[cfg_key])

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
for i, cfg_key in enumerate(configs):
    ax.plot(x, wins[cfg_key], "-o", color=colors[cfg_key], lw=2, ms=5, label=config_names[cfg_key])

ax.set_xlabel("tolerance γ", fontsize=12)
ax.set_ylabel("# instances where algorithm wins (smallest |S|)", fontsize=12)
ax.set_title(f"Algorithm wins per gamma  (total instances per gamma = {len(rows_at_g)})", fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Summary table

In [ ]:
# Print mean |S| table
print("mean |S| over seeds:")
hdr = f"{'gamma':>10} " + " ".join(f"{config_names[cfg_key]:>12}" for cfg_key in configs)
print(hdr)
print("-" * len(hdr))
for g in gammas:
    cells = [summary[cfg_key][g]["mean_size"] for cfg_key in configs]
    print(f"{g:10.4f} " + " ".join(f"{c:12.2f}" for c in cells))

# Overall wins
print("\nWins summary:")
for cfg_key in configs:
    total_wins = sum(wins[cfg_key])
    pct = 100 * total_wins / len(records)
    print(f"  {config_names[cfg_key]:>10} : {total_wins:>6} wins ({pct:>5.1f}%)")